In [ ]:
import pandas as pd
from helpers.pathlib import PATH_DATA

from ata_pipeline1.helpers.enums import Enum, FieldNew, FieldSnowplow
from ata_pipeline1.preprocessors import (
    AddFieldFormSubmitIsNewsletter,
    ConvertFieldFormSubmitToDataclass,
    ConvertFieldTypes,
)
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

CSV_PREFIX = "221103-230125"

In [ ]:
class FieldTemp(str, Enum):
    NEWSLETTER_EMAIL_ADDRESS = "newsletter_email_address"

In [ ]:
# Read data
df = pd.read_csv(PATH_DATA / f"{CSV_PREFIX}.csv")

In [ ]:
df = ConvertFieldTypes()(df)

In [ ]:
df = ConvertFieldFormSubmitToDataclass()(df)

In [ ]:
df_afla_0 = df.query(f"{FieldNew.SITE_NAME} == '{AFRO_LA.name}'")
df_dfp_0 = df.query(f"{FieldNew.SITE_NAME} == '{DALLAS_FREE_PRESS.name}'")
df_ov_0 = df.query(f"{FieldNew.SITE_NAME} == '{OPEN_VALLEJO.name}'")
df_19_0 = df.query(f"{FieldNew.SITE_NAME} == '{THE_19TH.name}'")

In [ ]:
df_afla_0 = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=AFRO_LA.newsletter_signup_validator)(
    df_afla_0
)
df_dfp_0 = AddFieldFormSubmitIsNewsletter(
    site_newsletter_signup_validator=DALLAS_FREE_PRESS.newsletter_signup_validator
)(df_dfp_0)
df_ov_0 = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=OPEN_VALLEJO.newsletter_signup_validator)(
    df_ov_0
)
df_19_0 = AddFieldFormSubmitIsNewsletter(site_newsletter_signup_validator=THE_19TH.newsletter_signup_validator)(df_19_0)

In [ ]:
# df_afla_7 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, AFRO_LA.name))
# df_dfp_7 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, DALLAS_FREE_PRESS.name))
# df_ov_7 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, OPEN_VALLEJO.name))
# df_19_7 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, THE_19TH.name))

In [ ]:
df_0 = pd.concat([df_afla_0, df_dfp_0, df_ov_0, df_19_0])
# df_7 = pd.concat([df_afla_7, df_dfp_7, df_ov_7, df_19_7])

## Form submissions in raw data

### Users

In [ ]:
# How many unique users?
df_0.groupby(FieldNew.SITE_NAME)[FieldSnowplow.DOMAIN_USERID].nunique()

### Newsletter form submissions

In [ ]:
# How many newsletter form submissions?
df_0.groupby(FieldNew.SITE_NAME)[FieldNew.FORM_SUBMIT_IS_NEWSLETTER].sum()

In [ ]:
# How many newsletter form submissions a subscriber (who submits the form at least once) makes at maximum?
df_0.groupby(FieldNew.SITE_NAME).apply(
    lambda df: (df.groupby(FieldSnowplow.DOMAIN_USERID)[FieldNew.FORM_SUBMIT_IS_NEWSLETTER].sum().astype(int).max())
)

In [ ]:
# How many newsletter form submissions a subscriber (who submits the form at least once) makes at maximum?
# (Same as above but top 10 instead of max)
df_0.groupby(FieldNew.SITE_NAME).apply(
    lambda df: (
        df.groupby(FieldSnowplow.DOMAIN_USERID)[FieldNew.FORM_SUBMIT_IS_NEWSLETTER]
        .sum()
        .astype(int)
        .sort_values(ascending=False)
        .head(10)
        .tolist()
    )
)

In [ ]:
# How many newsletter form submissions a subscriber (who submits the form at least once) makes on average?
df_0.groupby(FieldNew.SITE_NAME).apply(
    lambda df: (
        df.groupby(FieldSnowplow.DOMAIN_USERID)[FieldNew.FORM_SUBMIT_IS_NEWSLETTER]
        .sum()
        .astype(int)
        .where(lambda s: s >= 1)
        .mean()
    )
)

### Newsletter form submissions per session

In [ ]:
# How many newsletter form submissions a subscriber (who submits the form at least once) makes
# at maximum within a single session?
df_0.groupby(FieldNew.SITE_NAME).apply(
    lambda df: (
        df.groupby([FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX])[FieldNew.FORM_SUBMIT_IS_NEWSLETTER]
        .sum()
        .astype(int)
        .max()
    )
)

In [ ]:
# How many newsletter form submissions a subscriber (who submits the form at least once) makes
# at maximum within a single session?
# (Same as above but top 10)
df_0.groupby(FieldNew.SITE_NAME).apply(
    lambda df: (
        df.groupby([FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX])[FieldNew.FORM_SUBMIT_IS_NEWSLETTER]
        .sum()
        .astype(int)
        .sort_values(ascending=False)
        .head(10)
        .tolist()
    )
)

In [ ]:
# How many newsletter form submissions a subscriber (who submits the form at least once) makes
# on average within a single session?
df_0.groupby(FieldNew.SITE_NAME).apply(
    lambda df: (
        df.groupby([FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX])[FieldNew.FORM_SUBMIT_IS_NEWSLETTER]
        .sum()
        .astype(int)
        .where(lambda s: s >= 1)
        .mean()
    )
)

### Unique newsletter form submissions (determined via email address)

In [ ]:
# How many unique newsletter form submissions?
df_0.groupby(FieldNew.SITE_NAME)[FieldTemp.NEWSLETTER_EMAIL_ADDRESS].nunique()

In [ ]:
# How many unique newsletter form submissions a subscriber (who submits the form at least once) makes at maximum?
df_0.groupby(FieldNew.SITE_NAME).apply(
    lambda df: (df.groupby(FieldSnowplow.DOMAIN_USERID)[FieldTemp.NEWSLETTER_EMAIL_ADDRESS].nunique().max())
)

In [ ]:
# How many unique newsletter form submissions a subscriber (who submits the form at least once) makes at maximum?
# (Same as above but top 10 instead of max)
df_0.groupby(FieldNew.SITE_NAME).apply(
    lambda df: (
        df.groupby(FieldSnowplow.DOMAIN_USERID)[FieldTemp.NEWSLETTER_EMAIL_ADDRESS]
        .nunique()
        .sort_values(ascending=False)
        .head(10)
        .tolist()
    )
)

In [ ]:
# How many unique newsletter form submissions a subscriber (who submits the form at least once) makes on average?
df_0.groupby(FieldNew.SITE_NAME).apply(
    lambda df: (
        df.groupby(FieldSnowplow.DOMAIN_USERID)[FieldTemp.NEWSLETTER_EMAIL_ADDRESS]
        .nunique()
        .where(lambda x: x >= 1)
        .mean()
    )
)

### Unique newsletter form submissions per session

In [ ]:
# How many unique newsletter form submissions a subscriber (who submits the form at least once) makes
# at maximum within a single session?
# (Same as above but top 10)
df_0.groupby(FieldNew.SITE_NAME).apply(
    lambda df: (
        df.groupby([FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX])[FieldTemp.NEWSLETTER_EMAIL_ADDRESS]
        .nunique()
        .sort_values(ascending=False)
        .head(10)
        .tolist()
    )
)

In [ ]:
dummy = df_0.set_index([FieldNew.SITE_NAME, FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX]).query(
    f"{FieldNew.FORM_SUBMIT_IS_NEWSLETTER} == True"
)[[FieldNew.FORM_SUBMIT_IS_NEWSLETTER, FieldSnowplow.SEMISTRUCT_FORM_SUBMIT]]

In [ ]:
(
    dummy[FieldSnowplow.SEMISTRUCT_FORM_SUBMIT].apply(
        lambda x: [e for e in x["elements"] if e["type"] == "email"][0]["value"]
    )
)

In [ ]:
(
    df_0.query(f"{FieldNew.FORM_SUBMIT_IS_NEWSLETTER} == True")
    .groupby([FieldNew.SITE_NAME, FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX], group_keys=True)[
        FieldSnowplow.SEMISTRUCT_FORM_SUBMIT
    ]
    .agg(
        num_submissions="size",
        num_submissions_unique=lambda row: (
            row.apply(lambda x: [e for e in x["elements"] if e["type"] == "email"][0]["value"]).nunique()
        ),
    )
).sort_values("num_submissions_unique", ascending=False)

## The 19th* deep dive

In [ ]:
# All users' sessionidx where more than one newsletter form submission happened
df_19_dive = (
    df_19_0.groupby([FieldSnowplow.DOMAIN_USERID, FieldSnowplow.DOMAIN_SESSIONIDX])[FieldNew.FORM_SUBMIT_IS_NEWSLETTER]
    .sum()
    .astype(int)
    .where(lambda s: s > 2)
    .dropna()
    .astype(int)
    .sort_values(ascending=False)
    .rename("num_newsletter_submissions")
)

In [ ]:
df_19_dive

In [ ]:
dive_users = df_19_dive.index.get_level_values(0).unique().tolist()

In [ ]:
df_19_0.query(f"{FieldSnowplow.DOMAIN_USERID} == 'e145572d-0f8d-446f-be47-42e5602dbdf2'").useragent.iloc[0]

In [ ]:
df_19_0.query(f"{FieldSnowplow.DOMAIN_USERID} == '0a1b9377-cc5a-4763-a1fb-13e77db57335'").useragent.iloc[0]

In [ ]:
df_19_0.query(f"{FieldSnowplow.DOMAIN_USERID} == 'b7878956-1771-4963-816c-e05967a44118'").useragent.iloc[0]

In [ ]:
df_19_0.query(f"{FieldSnowplow.DOMAIN_USERID} == '28bc93f9-1a55-4803-9193-ea7c03a9ef8c'").useragent.iloc[0]

In [ ]:
df_19_0.query(f"{FieldSnowplow.DOMAIN_USERID} == '6ba9d4f3-97cb-4f71-8503-8a8c4ea059e6'").useragent.iloc[0]

In [ ]:
def get_email_addresses_repeated_form_submits(df, domain_userids):
    return (
        df.query(
            f"{FieldNew.SITE_NAME} == '{THE_19TH.name}' and "
            + f"{FieldSnowplow.DOMAIN_USERID}.isin({domain_userids}) and "
            + f"{FieldSnowplow.EVENT_NAME} == 'submit_form'"
        )[FieldSnowplow.SEMISTRUCT_FORM_SUBMIT]
        .apply(lambda x: [e for e in x["elements"] if e["type"] == "email"][0]["value"])
        .value_counts()
    )

In [ ]:
get_email_addresses_repeated_form_submits(df_19_0, ["ffb03b18-299b-4c71-bb54-ebdb2536f984"])